In [1]:
from pathlib import Path

import requests

import pandas as pd

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent


In [ ]:
year = [2023, 2022, 2021, 2020, 2019, 2018, 2017, 2016, 2015]
url = "https://www.gov.br/mj/pt-br/assuntos/sua-seguranca/seguranca-publica/estatistica/download/dnsp-base-de-dados/bancovde-{y}.xlsx/@@download/file"
download_path = project_root / "data" / "xlsx_dowloads"

In [3]:
def download_xlsx(url: str, year: int, download_dir: str | Path) -> Path:
    download_dir = Path(download_dir)
    download_dir.mkdir(parents=True, exist_ok=True)

    download_url = url.format(y=year)
    file_path = download_dir / f"pubseg-{year}.xlsx"
    response = requests.get(download_url, timeout=60)
    response.raise_for_status()
    file_path.write_bytes(response.content)

    return file_path

In [4]:
import pandas as pd
from sqlalchemy import (
    BigInteger,
    Boolean,
    Column,
    DateTime,
    Float,
    MetaData,
    String,
    Table,
    Text,
    create_engine,
    inspect,
)
from sqlalchemy.engine import URL
from sqlalchemy.schema import CreateSchema


def build_postgres_engine(host: str, port: int, database: str, user: str, password: str):
    return create_engine(
        URL.create(
            "postgresql+psycopg2",
            username=user,
            password=password,
            host=host,
            port=port,
            database=database,
        )
    )


def pandas_dtype_to_sqlalchemy(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return BigInteger
    if pd.api.types.is_float_dtype(dtype):
        return Float
    if pd.api.types.is_bool_dtype(dtype):
        return Boolean
    if pd.api.types.is_datetime64_any_dtype(dtype):
        return DateTime
    return Text


def create_table_from_dataframe(
    dataframe: pd.DataFrame,
    host: str,
    port: int,
    database: str,
    user: str,
    password: str,
    schema_name: str,
    table_name: str,
) -> Table:
    if "id_tabela" not in dataframe.columns:
        raise KeyError("O dataframe precisa ter a coluna 'id_tabela'.")

    metadata = MetaData(schema=schema_name)
    columns = [Column("id", String, primary_key=True)]
    for column_name, dtype in dataframe.dtypes.items():
        columns.append(
            Column(
                str(column_name),
                pandas_dtype_to_sqlalchemy(dtype),
                nullable=bool(dataframe[column_name].isna().any()),
            )
        )

    table = Table(table_name, metadata, *columns)
    engine = build_postgres_engine(host, port, database, user, password)

    with engine.begin() as connection:
        if not inspect(connection).has_schema(schema_name):
            connection.execute(CreateSchema(schema_name))
        metadata.create_all(connection, tables=[table], checkfirst=True)

    return table

In [5]:
def upload_dataframe_to_postgres(
    dataframe: pd.DataFrame,
    host: str,
    port: int,
    database: str,
    user: str,
    password: str,
    schema_name: str,
    table_name: str,
) -> int:
    if "id_tabela" not in dataframe.columns:
        raise KeyError("O dataframe precisa ter a coluna 'id_tabela'.")
    if dataframe["id_tabela"].isna().any():
        raise ValueError("A coluna 'id_tabela' não pode ter valores nulos.")

    frame = dataframe.copy()
    id_series = frame["id_tabela"].astype("string").str.replace(r"\.0+$", "", regex=True)
    frame.insert(0, "id", id_series)

    engine = build_postgres_engine(host, port, database, user, password)
    with engine.begin() as connection:
        frame.to_sql(
            table_name,
            con=connection,
            schema=schema_name,
            if_exists="append",
            index=False,
            method="multi",
        )

    return len(frame)

In [6]:
from pathlib import Path

from dotenv import dotenv_values

env = dotenv_values(project_root / ".env")
host = "localhost"
port = int(env.get("POSTGRES_PORT", 5432))
database = env["POSTGRES_DB"]
user = env["POSTGRES_USER"]
password = env["POSTGRES_PASSWORD"]
schema_name = "public"
table_name = "pubseg"

from sqlalchemy import text


def drop_table(schema_name: str, table_name: str) -> None:
    engine = build_postgres_engine(host, port, database, user, password)

    with engine.begin() as connection:
        connection.execute(text(f'DROP TABLE IF EXISTS \"{schema_name}\".\"{table_name}\"'))

drop_table(schema_name, table_name)


In [7]:
def table_ajust(db: pd.DataFrame):

    year_series = pd.to_datetime(db["data_referencia"], errors="coerce").dt.year

    if year_series.isna().any():
        raise ValueError("The 'data_referencia' column must contain valid dates to generate the id_tabela.")
    
    row_id = pd.Series(range(1, len(db) + 1), index=db.index, dtype="int64")
    db["id_tabela"] = year_series.astype("Int64").astype("string") + row_id.astype("string")
    ordered_columns = ["id_tabela", *[column for column in db.columns if column != "id_tabela"]]
    db = db[ordered_columns]

    db["feminino"] = (
        pd.to_numeric(db["feminino"], errors="coerce")  # garante numérico
        .fillna(0)                                     # NaN → 0
        .ne(0)                                         # != 0 → True
    )
    db["masculino"] = (
        pd.to_numeric(db["masculino"], errors="coerce")  # garante numérico
        .fillna(0)                                     # NaN → 0
        .ne(0)                                         # != 0 → True
    )
    db["nao_informado"] = (
        pd.to_numeric(db["nao_informado"], errors="coerce")  # garante numérico
        .fillna(0)                                     # NaN → 0
        .ne(0)                                         # != 0 → True
    )
    db["total_vitima"] = (
        pd.to_numeric(db["total_vitima"], errors="coerce")
        .fillna(0)
        .astype(int)
    )

    return db


Se não for a primeira vez rodando o script pode comentar essa parte

In [8]:
path = download_xlsx(url, year[0], download_path)
db = pd.read_excel(path)
db = table_ajust(db)
create_table_from_dataframe(
    db,
    host=host,
    port=port,
    database=database,
    user=user,
    password=password,
    schema_name=schema_name,
    table_name=table_name,
)

Table('pubseg', MetaData(), Column('id', String(), table=<pubseg>, primary_key=True, nullable=False), Column('id_tabela', Text(), table=<pubseg>, nullable=False), Column('uf', Text(), table=<pubseg>, nullable=False), Column('municipio', Text(), table=<pubseg>, nullable=False), Column('evento', Text(), table=<pubseg>, nullable=False), Column('data_referencia', DateTime(), table=<pubseg>, nullable=False), Column('agente', Text(), table=<pubseg>), Column('arma', Text(), table=<pubseg>), Column('faixa_etaria', Text(), table=<pubseg>), Column('feminino', Boolean(), table=<pubseg>, nullable=False), Column('masculino', Boolean(), table=<pubseg>, nullable=False), Column('nao_informado', Boolean(), table=<pubseg>, nullable=False), Column('total_vitima', BigInteger(), table=<pubseg>, nullable=False), Column('total', Float(), table=<pubseg>), Column('total_peso', Float(), table=<pubseg>), Column('abrangencia', Text(), table=<pubseg>, nullable=False), schema='public')

In [9]:

def dowload_full_db(url:str, year:list[int], download_dir: str | Path, 
                    schema_name:str, table_name:str):
    for y in year:
        try:
            path = download_xlsx(url, y, download_dir)
            db = pd.read_excel(path)
            db = table_ajust(db)

        except Exception as e:
            print(f"error during {y} data dowload: {e}")
            continue
        try:
            rows_inserted = upload_dataframe_to_postgres(
                db,
                host=host,
                port=port,
                database=database,
                user=user,
                password=password,
                schema_name=schema_name,
                table_name=table_name,
            )

            print(f"Lines inserted in {schema_name}.{table_name}: {rows_inserted}, from {y} data")

        except:
            raise


        

In [10]:
dowload_full_db(url, year, download_path, schema_name, table_name)

Lines inserted in public.pubseg: 832046, from 2025 data
Lines inserted in public.pubseg: 764508, from 2024 data
error during 2023 data dowload: ('Connection broken: IncompleteRead(23343643 bytes read, 6719393 more expected)', IncompleteRead(23343643 bytes read, 6719393 more expected))


In [11]:
query = f"SELECT * FROM {schema_name}.{table_name} LIMIT 100"

with build_postgres_engine(host, port, database, user, password).connect() as connection:
    preview_db = pd.read_sql(query, connection)

preview_db

,id,id_tabela,uf,municipio,evento,data_referencia,agente,arma,faixa_etaria,feminino,masculino,nao_informado,total_vitima,total,total_peso,abrangencia
0,20251,20251,AC,ACRELÂNDIA,Feminicídio,2025-01-01,None,None,None,False,False,False,0,NaN,None,Estadual
1,20252,20252,AC,ACRELÂNDIA,Feminicídio,2025-02-01,None,None,None,False,False,False,0,NaN,None,Estadual
2,20253,20253,AC,ACRELÂNDIA,Feminicídio,2025-03-01,None,None,None,False,False,False,0,NaN,None,Estadual
3,20254,20254,AC,ACRELÂNDIA,Feminicídio,2025-04-01,None,None,None,False,False,False,0,NaN,None,Estadual
4,20255,20255,AC,ACRELÂNDIA,Feminicídio,2025-05-01,None,None,None,False,False,False,0,NaN,None,Estadual
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,202596,202596,AC,ACRELÂNDIA,Mortes no trânsito,2025-12-01,None,None,None,False,False,False,0,NaN,None,Polícia Rodoviária Federal
96,202597,202597,AC,ACRELÂNDIA,Roubo seguido de morte (latrocínio),2025-01-01,None,None,None,False,False,False,0,NaN,None,Estadual
97,202598,202598,AC,ACRELÂNDIA,Roubo seguido de morte (latrocínio),2025-02-01,None,None,None,False,False,False,0,NaN,None,Estadual
98,202599,202599,AC,ACRELÂNDIA,Roubo seguido de morte (latrocínio),2025-03-01,None,None,None,False,False,False,0,NaN,None,Estadual
